In [2]:
import requests
import pandas as pd
import time

BASE_URL_PLAYERS = "https://compstats.uefa.com/v1/player-ranking"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; ucl-research-bot/2.0)"
}

def safe_int(x):
    try:
        return int(x)
    except:
        return None


def get_player_stats_one_season(season_year: int,
                                stat_names: list,
                                limit: int = 400,
                                offset: int = 0) -> pd.DataFrame:

    params = {
        "competitionId": "1",         # Champions League
        "limit": str(limit),          # máximo de jugadores
        "offset": str(offset),
        "optionalFields": "PLAYER,TEAM",
        "order": "DESC",
        "phase": "TOURNAMENT",
        "seasonYear": str(season_year),
        "stats": ",".join(stat_names)
    }

    r = requests.get(BASE_URL_PLAYERS, params=params,
                     headers=HEADERS, timeout=15)
    r.raise_for_status()

    data = r.json()  # lista con info de muchos jugadores

    rows = []

    for entry in data:

        player = entry.get("player", {})
        team = entry.get("team", {})
        stats_list = entry.get("statistics", [])

        stats_dict = {s["name"]: s["value"] for s in stats_list}

        translations = player.get("translations", {})
        display_name = translations.get("displayName", {})
        pos_name = translations.get("positionName", {})

        rows.append({
            "season_year": season_year,
            "player_id": entry.get("playerId"),
            "player_name_en": display_name.get("EN"),
            "player_name_es": display_name.get("ES"),
            "position_en": pos_name.get("EN"),
            "position_es": pos_name.get("ES"),
            "team_id": team.get("teamId"),
            "team_code": team.get("teamCode"),

            # Goles (general)
            "goals_total": safe_int(stats_dict.get("goals_scored")),

            # Detalles de goles
            "goals_right_foot": safe_int(stats_dict.get("goals_right_foot")),
            "goals_left_foot": safe_int(stats_dict.get("goals_left_foot")),
            "goals_head": safe_int(stats_dict.get("goals_head")),
            "goals_outside_box": safe_int(stats_dict.get("goals_outside_box")),
            "goals_inside_box": safe_int(stats_dict.get("goals_inside_box")),
            "goals_penalty": safe_int(stats_dict.get("goals_penalty")),

            # Disparos
            "shots_on_target": safe_int(stats_dict.get("shots_on_target")),
            "shots_total": safe_int(stats_dict.get("shots")),
            "shots_outside_box": safe_int(stats_dict.get("shots_outside_box")),
            "shots_inside_box": safe_int(stats_dict.get("shots_inside_box")),

            # Expected goals
            "xg_total": stats_dict.get("expected_goals"),
        })

    return pd.DataFrame(rows)



if __name__ == "__main__":

    # Las estadísticas que quieres
    stats_wanted = [
        "goals_scored", "goals_right_foot", "goals_left_foot", "goals_head",
        "goals_outside_box", "goals_inside_box", "goals_penalty",
        "shots_on_target", "shots", "shots_outside_box", "shots_inside_box",
        "expected_goals"
    ]

    all_dfs = []

    for season in range(1992, 2026):
        print(f"🟦 Descargando jugadores {season}/{season+1}...")
        try:
            df = get_player_stats_one_season(season, stats_wanted)
            if not df.empty:
                all_dfs.append(df)
        except Exception as e:
            print(f"❌ Error en temporada {season}: {e}")

        time.sleep(1)

    if all_dfs:
        players_df = pd.concat(all_dfs, ignore_index=True)
        players_df.to_csv("ucl_players_attacking_stats_1992_2025.csv", index=False)
        print("✅ Guardado: ucl_players_attacking_stats_1992_2025.csv")
    else:
        print("❌ No se descargó nada.")


🟦 Descargando jugadores 1992/1993...
🟦 Descargando jugadores 1993/1994...
🟦 Descargando jugadores 1994/1995...
🟦 Descargando jugadores 1995/1996...
🟦 Descargando jugadores 1996/1997...
🟦 Descargando jugadores 1997/1998...
🟦 Descargando jugadores 1998/1999...
🟦 Descargando jugadores 1999/2000...
🟦 Descargando jugadores 2000/2001...
🟦 Descargando jugadores 2001/2002...
🟦 Descargando jugadores 2002/2003...
🟦 Descargando jugadores 2003/2004...
🟦 Descargando jugadores 2004/2005...
🟦 Descargando jugadores 2005/2006...
🟦 Descargando jugadores 2006/2007...
🟦 Descargando jugadores 2007/2008...
🟦 Descargando jugadores 2008/2009...
🟦 Descargando jugadores 2009/2010...
🟦 Descargando jugadores 2010/2011...
🟦 Descargando jugadores 2011/2012...
🟦 Descargando jugadores 2012/2013...
🟦 Descargando jugadores 2013/2014...
🟦 Descargando jugadores 2014/2015...
🟦 Descargando jugadores 2015/2016...
🟦 Descargando jugadores 2016/2017...
🟦 Descargando jugadores 2017/2018...
🟦 Descargando jugadores 2018/2019...
🟦

In [6]:
import requests

def get_players(season_year):
    url = "https://compstats.uefa.com/v1/player-ranking"
    params = {
        "competitionId": "1",
        "seasonYear": str(season_year),
        "phase": "TOURNAMENT",
        "limit": "300",       # suficiente para toda la UCL
        "offset": "0",
        "optionalFields": "PLAYER,TEAM"
    }
    headers = {"User-Agent": "Mozilla/5.0"}

    r = requests.get(url, params=params, headers=headers)
    r.raise_for_status()
    data = r.json()

    players = []
    for entry in data:
        player = entry.get("player", {})
        team = entry.get("team", {})
        players.append({
            "playerId": entry.get("playerId"),
            "name_en": player.get("translations", {}).get("displayName", {}).get("EN"),
            "team": team.get("teamCode")
        })
    return players


In [7]:
def get_player_stats(player_id, season_year):
    url = f"https://match.uefa.com/v3/players/{player_id}/statistics"
    params = {
        "competitionId": "1",
        "seasonYear": str(season_year)
    }
    headers = {"User-Agent": "Mozilla/5.0"}

    r = requests.get(url, params=params, headers=headers)
    if r.status_code != 200:
        return None
    return r.json()


In [9]:
import requests
import pandas as pd
import time

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; ucl-research-bot/2.0)"
}

# ================================
# 1) Obtener lista de jugadores
# ================================
def get_players_list(season_year: int, limit: int = 400):
    """
    Devuelve lista de jugadores (playerId + info básica) para una temporada.
    ESTE ENDPOINT SOLO FUNCIONA DESDE 2017.
    """
    url = "https://compstats.uefa.com/v1/player-ranking"
    params = {
        "competitionId": "1",
        "seasonYear": str(season_year),
        "phase": "TOURNAMENT",
        "limit": str(limit),
        "offset": "0",
        "optionalFields": "PLAYER,TEAM"
    }

    r = requests.get(url, params=params, headers=HEADERS, timeout=15)
    r.raise_for_status()
    data = r.json()

    players = []
    for entry in data:
        p = entry.get("player", {})
        t = entry.get("team", {})
        translations = p.get("translations", {})
        display = translations.get("displayName", {})

        players.append({
            "playerId": entry.get("playerId"),
            "player_name_en": display.get("EN"),
            "team_code": t.get("teamCode"),
        })

    return players


# ================================
# 2) Obtener estadísticas completas de un jugador
# ================================
def get_player_stats(player_id: int, season_year: int):
    """
    Descarga TODAS las estadísticas públicas del jugador.
    """
    url = f"https://match.uefa.com/v3/players/{player_id}/statistics"
    params = {
        "competitionId": "1",
        "seasonYear": str(season_year)
    }

    r = requests.get(url, params=params, headers=HEADERS, timeout=15)
    if r.status_code != 200:
        return None

    return r.json()


# ================================
# 3) Scraping completo 2017–2025
# ================================
if __name__ == "__main__":
    all_rows = []

    for season in range(2017, 2026):
        print(f"\n🟦 Temporada {season}/{season+1}…")

        try:
            players = get_players_list(season)
        except Exception as e:
            print(f"  ⚠️ Error obteniendo jugadores ({season}): {e}")
            continue

        for p in players:
            pid = p["playerId"]

            stats_json = get_player_stats(pid, season)
            if stats_json is None:
                continue

            # Aplanar JSON de stats
            flat_stats = {}
            for category in stats_json.get("statistics", []):
                for stat in category.get("events", []):
                    stat_name = stat.get("name")
                    stat_value = stat.get("value")
                    flat_stats[stat_name] = stat_value

            row = {
                "season_year": season,
                "playerId": pid,
                "player_name_en": p["player_name_en"],
                "team_code": p["team_code"],
            }
            row.update(flat_stats)

            all_rows.append(row)
            time.sleep(0.2)  # para no saturar servidores

    df = pd.DataFrame(all_rows)
    df.to_csv("ucl_players_all_stats_2017_2025.csv", index=False)
    print("\n📁 Archivo guardado: ucl_players_all_stats_2017_2025.csv")



🟦 Temporada 2017/2018…
  ⚠️ Error obteniendo jugadores (2017): 400 Client Error: Bad Request for url: https://compstats.uefa.com/v1/player-ranking?competitionId=1&seasonYear=2017&phase=TOURNAMENT&limit=400&offset=0&optionalFields=PLAYER%2CTEAM

🟦 Temporada 2018/2019…
  ⚠️ Error obteniendo jugadores (2018): 400 Client Error: Bad Request for url: https://compstats.uefa.com/v1/player-ranking?competitionId=1&seasonYear=2018&phase=TOURNAMENT&limit=400&offset=0&optionalFields=PLAYER%2CTEAM

🟦 Temporada 2019/2020…
  ⚠️ Error obteniendo jugadores (2019): 400 Client Error: Bad Request for url: https://compstats.uefa.com/v1/player-ranking?competitionId=1&seasonYear=2019&phase=TOURNAMENT&limit=400&offset=0&optionalFields=PLAYER%2CTEAM

🟦 Temporada 2020/2021…
  ⚠️ Error obteniendo jugadores (2020): 400 Client Error: Bad Request for url: https://compstats.uefa.com/v1/player-ranking?competitionId=1&seasonYear=2020&phase=TOURNAMENT&limit=400&offset=0&optionalFields=PLAYER%2CTEAM

🟦 Temporada 2021/20